# Kémzy àvátâr — PersonaLive CUDA + Video Render Proof

This notebook proves the **real Kémzy PersonaLive renderer** on a Kaggle CUDA GPU. Large weights remain in Kaggle and are never downloaded to the Android phone.

The test path is: **GPU → PersonaLive pipeline → reference image → real driving frames → generated neural frame**.

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA GPU is required for this proof.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

In [ ]:
!rm -rf /kaggle/working/Kemzy-LiveAvatar /kaggle/working/PersonaLive
!git clone --depth 1 --branch feature/backend-render-gateway https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git /kaggle/working/Kemzy-LiveAvatar
!git clone --depth 1 --branch abdd112e01dcf7d89122c2e5efa29fcff0669740 https://github.com/GVCLab/PersonaLive.git /kaggle/working/PersonaLive
%cd /kaggle/working/PersonaLive
!pip install -q -r requirements_base.txt

In [ ]:
# Official PersonaLive weights stay in Kaggle only.
!python tools/download_weights.py

In [ ]:
import os, shutil
src='/kaggle/working/Kemzy-LiveAvatar/backend/renderer/personalive_server.py'
dst='/kaggle/working/PersonaLive/personalive_server.py'
shutil.copy2(src,dst)
print('Kémzy server:', dst)
print('Endpoint: POST /v1/diagnostics/render')

## Upload the source and optional driving video

Upload a clear portrait as the reference. If you also upload a short driving video, the notebook extracts four real consecutive frames. This is stronger than using brightness-only synthetic frames.

In [ ]:
from IPython.display import display
from ipywidgets import FileUpload
print('Reference portrait:')
reference_upload=FileUpload(accept='image/*', multiple=False)
display(reference_upload)
print('Optional driving video:')
video_upload=FileUpload(accept='video/*', multiple=False)
display(video_upload)

In [ ]:
from PIL import Image
import io, os, subprocess, json
assert reference_upload.value, 'Upload a reference portrait first.'
item=next(iter(reference_upload.value.values()))
Image.open(io.BytesIO(item['content'])).convert('RGB').save('/kaggle/working/reference.jpg', quality=95)
if video_upload.value:
    v=next(iter(video_upload.value.values()))
    video_bytes=v['content']
    open('/kaggle/working/driving.mp4','wb').write(video_bytes)
    probe=subprocess.run(['ffprobe','-v','error','-show_entries','stream=width,height,r_frame_rate,nb_frames','-of','json','/kaggle/working/driving.mp4'],capture_output=True,text=True)
    print('Video probe:', probe.stdout)
    subprocess.run(['ffmpeg','-y','-i','/kaggle/working/driving.mp4','-vf','select=not(mod(n\,4))','-frames:v','4','/kaggle/working/frame%01d.jpg'],check=True)
else:
    img=Image.open('/kaggle/working/reference.jpg').convert('RGB')
    # Fallback smoke test only: valid frames, but not real motion.
    for i in range(1,5): img.save(f'/kaggle/working/frame{i}.jpg', quality=92)
    print('No driving video supplied; using the reference as a fallback smoke test.')
assert all(os.path.exists(f'/kaggle/working/frame{i}.jpg') for i in range(1,5)), 'Four driving frames were not created.'
print('Reference bytes:', os.path.getsize('/kaggle/working/reference.jpg'))
print('Driving frames:', [os.path.getsize(f'/kaggle/working/frame{i}.jpg') for i in range(1,5)])

In [ ]:
%cd /kaggle/working/PersonaLive
import subprocess, os, time
env=os.environ.copy()
env.update({'MODEL_DIR':'/kaggle/working/PersonaLive','PERSONALIVE_CONFIG':'/kaggle/working/PersonaLive/configs/prompts/personalive_online.yaml','ACCELERATION':'none','DIAGNOSTIC_RENDER_TIMEOUT':'180'})
server=subprocess.Popen(['python','-m','uvicorn','personalive_server:app','--host','0.0.0.0','--port','7860'],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
time.sleep(10)
health=subprocess.run(['curl','-s','http://127.0.0.1:7860/health'],capture_output=True,text=True)
print('server pid:',server.pid)
print('health:',health.stdout)

In [ ]:
import requests, base64, os
paths={'reference':'/kaggle/working/reference.jpg','frame1':'/kaggle/working/frame1.jpg','frame2':'/kaggle/working/frame2.jpg','frame3':'/kaggle/working/frame3.jpg','frame4':'/kaggle/working/frame4.jpg'}
handles=[]
files={}
try:
    for field,path in paths.items():
        f=open(path,'rb'); handles.append(f); files[field]=(os.path.basename(path),f,'image/jpeg')
    r=requests.post('http://127.0.0.1:7860/v1/diagnostics/render',files=files,timeout=240)
    print('HTTP:',r.status_code)
    print(r.text[:4000])
    r.raise_for_status()
    result=r.json()
    jpeg=base64.b64decode(result['first_frame_jpeg_base64'])
    open('/kaggle/working/kemzy_personalive_render.jpg','wb').write(jpeg)
    print('Verified generated JPEG bytes:',len(jpeg))
finally:
    for f in handles: f.close()

In [ ]:
from IPython.display import display
display(Image.open('/kaggle/working/kemzy_personalive_render.jpg'))

## Evidence checklist

- CUDA GPU detected
- Exact upstream PersonaLive commit used
- Official weights downloaded only inside Kaggle
- Kémzy server loaded
- Reference fused
- Four driving frames submitted
- `generated_frame_count` returned by the real render endpoint
- Generated JPEG decoded and displayed

If a real driving video was supplied, the four submitted frames came from that video. This is the next proof step toward the Android persistent camera → neural renderer → preview pipeline.